In [ ]:
import pandas as pd
import numpy as np


# ** 1. Load and inspect dataset**

In [ ]:
df=pd.read_csv("/content/OnlineRetail.csv",encoding='latin-1')

In [ ]:
df.info()

# **Inspect the dataset**

In [ ]:
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df.head(100)

In [ ]:
df.tail(100)

In [ ]:

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nZero or negative quantities:", (df['Quantity'] <= 0).sum())

print("\nZero or negative prices:", (df['UnitPrice'] <= 0).sum())

print("\nUnique invoices:", df['InvoiceNo'].nunique())
print("Unique customers:", df['CustomerID'].nunique())
print("Unique products:", df['StockCode'].nunique())

# 6. Basic statistics
df.describe(include='all')


# **3. Clean missing and duplicate data**


In [ ]:
df_clean=df.copy()

In [ ]:
df_clean.duplicated().sum()

In [ ]:
df_clean.drop_duplicates(inplace=True)
df_clean.duplicated().sum()

In [ ]:
df_clean['Description'].fillna('Unknown', inplace=True)

In [ ]:
print("\nMissing CustomerID values:",df_clean['CustomerID'].isna().sum())
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

print("\nLen of original data df: ",len(df))
print("\nLen of cleaned data df_clean: ",len(df_clean))
print("\nNumber of rows removed : ",len(df)-len(df_clean))

# **4. Fix data types and text**

In [ ]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'],errors='coerce')
df_clean['CustomerID'] = df_clean['CustomerID'].astype('Int64')
df_clean['Description'] = (df_clean['Description'].astype('string').str.strip())
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype('string')
df_clean['StockCode'] = df_clean['StockCode'].astype('string')
print(df_clean.dtypes)

In [ ]:
print("\nMissing InvoiceDate values:",df_clean['InvoiceDate'].isna().sum())

In [ ]:
display(df_clean.head())

In [ ]:
print("Zero quantities:",(df_clean['Quantity'] == 0).sum())

print("Negative quantities:",(df_clean['Quantity'] < 0).sum())

print("\nZero prices:",(df_clean['UnitPrice'] == 0).sum())

print("Negative prices:",(df_clean['UnitPrice'] < 0).sum())

In [ ]:
print("\nNegative quantity records:")
df_clean[df_clean['Quantity'] < 0][['InvoiceNo', 'StockCode', 'Description','Quantity', 'UnitPrice', 'CustomerID']].head(10)

# **5. Handle invalid values and outliers**

In [ ]:
print("\nNegative price records:")
df_clean[df_clean['UnitPrice'] <= 0][['InvoiceNo', 'StockCode', 'Description','Quantity', 'UnitPrice', 'CustomerID']].head(10)

In [ ]:
#add a column or Tital_Price= Quantity*Unit Price

df_clean['TotalAmount'] = (df_clean['Quantity']*df_clean['UnitPrice'])

In [ ]:
print("Smallest transaction values:")
print(df_clean.nsmallest(10, 'TotalAmount'))

print("\nLargest transaction values:")
print(df_clean.nlargest(10, 'TotalAmount'))

In [ ]:

# Investigate extreme quantity records
extreme_quantities = df_clean[df_clean['Quantity'].abs() > 10000]

display(extreme_quantities.sort_values(['StockCode', 'InvoiceDate']))

# Investigate special non-product records
special_records = df_clean[df_clean['StockCode'].isin(['AMAZONFEE', 'M', 'B'])| df_clean['Description'].str.contains('bad debt|manual|amazon fee',case=False,na=False)]

display(special_records.head(20))


In [ ]:
print("Special records found:", len(special_records))

In [ ]:

# Categorize special records for investigation

special_records = df_clean[df_clean['StockCode'].isin(['AMAZONFEE', 'M', 'B'])| df_clean['Description'].str.contains('bad debt|manual|amazon fee',case=False,na=False)]

print("Total special records:", len(special_records))

print("\nRecords by StockCode:")
display(special_records['StockCode'].value_counts(dropna=False).to_frame('Count'))

print("\nRecords by Description:")
display(special_records['Description'].value_counts(dropna=False).head(15).to_frame('Count'))


In [ ]:
# Count zero and negative prices
print("Zero prices:",(df_clean['UnitPrice'] == 0).sum())

print("Negative prices:",(df_clean['UnitPrice'] < 0).sum())

# Inspect zero-priced transactions
zero_prices = df_clean[df_clean['UnitPrice'] == 0]

print("\nZero-priced records:", len(zero_prices))
display(zero_prices[['InvoiceNo', 'StockCode', 'Description','Quantity', 'UnitPrice', 'CustomerID', 'Country']].head(15))

# Inspect negative-priced transactions
negative_prices = df_clean[df_clean['UnitPrice'] < 0]

print("\nNegative-priced records:", len(negative_prices))
display(negative_prices[['InvoiceNo', 'StockCode', 'Description','Quantity', 'UnitPrice', 'CustomerID', 'Country']].head(15))


In [ ]:

# Count negative, zero, and positive values

for col in ['Quantity', 'UnitPrice']:
    print(f"\n--- {col} ---")
    print("Negative values:", (df_clean[col] < 0).sum())
    print("Zero values:", (df_clean[col] == 0).sum())
    print("Positive values:", (df_clean[col] > 0).sum())


In [ ]:

# Inspect all records with negative UnitPrice
df_clean[df_clean['UnitPrice'] < 0]



In [ ]:

# Investigate zero-priced records

zero_price = df_clean[df_clean['UnitPrice'] == 0].copy()

print("Total zero-priced records:", len(zero_price))

print("\nDescriptions of zero-priced records:")
display(
    zero_price['Description']
    .value_counts(dropna=False)
    .head(20)
    .to_frame('Count')
)

print("\nSample zero-priced transactions:")
display(
    zero_price[
        ['InvoiceNo', 'StockCode', 'Description',
         'Quantity', 'UnitPrice', 'CustomerID', 'Country']
    ].head(20)
)


In [ ]:

# Categorize zero-priced records for investigation

zero_price = df_clean[df_clean['UnitPrice'] == 0].copy()

# Check whether invoice numbers start with C
zero_price['IsCancellation'] = (
    zero_price['InvoiceNo'].str.startswith('C', na=False)
)

print("Zero-priced records with cancellation invoices:")
print(zero_price['IsCancellation'].value_counts())

# Check quantities
print("\nQuantity distribution:")
print(zero_price['Quantity'].describe())

print("\nNegative quantities among zero-priced records:",
      (zero_price['Quantity'] < 0).sum())

# Inspect records with descriptions suggesting an issue
issue_keywords = (
    r'unknown|check|damage|adjustment|found|destroyed|'
    r'thrown away|amazon|\?'
)

zero_price['PossibleIssue'] = zero_price['Description'].str.contains(
    issue_keywords,
    case=False,
    na=False,
    regex=True
)

print("\nRecords with potentially non-standard descriptions:",
      zero_price['PossibleIssue'].sum())

display(
    zero_price[
        ['InvoiceNo', 'StockCode', 'Description',
         'Quantity', 'UnitPrice', 'CustomerID',
         'IsCancellation', 'PossibleIssue']
    ].head(30)
)


In [ ]:
# Flag negative quantities instead of deleting them
df_clean['IsReturnOrCancellation'] = df_clean['Quantity'] < 0

# Check the counts
print(df_clean['IsReturnOrCancellation'].value_counts())

# **6. Validate and export**

In [ ]:

# Step 6: Final validation

print("Final dataset shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("\nData types:")
print(df_clean.dtypes)

# Check the transaction amount column
df_clean['TotalAmount'] = (
    df_clean['Quantity'] * df_clean['UnitPrice']
)

print("\nSample cleaned records:")
display(df_clean.head())

print("\nValidation completed!")


In [ ]:
df_clean.info()

## **MISSING CUSTOMER ID**

In [ ]:
df_clean['CustomerID'] = (
    df_clean['CustomerID'].astype('Int64')
)

print("Missing CustomerIDs:",
      df_clean['CustomerID'].isna().sum())

### **Data Cleaning Progress Checklist**

- <input type="checkbox" checked> **1. Load and inspect dataset**
- <input type="checkbox" checked> **2. Assess data quality**
- <input type="checkbox" checked> **3. Clean missing and duplicate data**
- <input type="checkbox"> **4. Fix data types and text**
- <input type="checkbox"> **5. Handle invalid values and outliers**
- <input type="checkbox"> **6. Validate and export cleaned data**